# DailyDialog → reviewed-data Qwen reranker

**Run this notebook yourself on Kaggle.** Attach the supplied private input dataset, enable Internet, and select one GPU (T4 is sufficient). Keep the notebook and input dataset private. No account credentials are needed in these cells.

Stage 1 learns reply suitability from human-written DailyDialog turns and weak mismatches. Stage 2 continues the selected adapter on our reviewed training bundle. This ranks existing replies; it does not train a new reply generator. Existing checkpoints and demo defaults are unchanged.

DailyDialog speaker roles assume alternating **utterance turns**. Our project queries retain their supplied roles, including repeated speakers. No AI-generated dialogue, AI ratings or project test items are included. DailyDialog uses CC BY-NC-SA 4.0; cite Li et al. (2017).

29 October fix: bounded AMP loss-scale recovery retries the same batch after gradient overflow. Both stages keep strict finite-gradient clipping. Use the latest private input version.


In [ ]:
import subprocess, sys
# This experiment does not use Kaggle's optional torchao quantization package.
subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet",
                       "transformers==5.19.0", "tokenizers==0.23.2",
                       "safetensors==0.8.0", "peft==0.21.2"])


In [ ]:
# Settings: change RUN_NAME for each new run; existing results are never overwritten.
RUN_NAME = "qwen_dailydialog_03"
MAX_TRAIN_QUERIES = 4000       # up to 12,000 pairs: one observed reply + two weak negatives
MAX_VALIDATION_QUERIES = 400 # up to 1,200 pairs, separate full dialogues
ADAPTATION_EPOCHS = 2         # stage 1; patience 1
REFINEMENT_EPOCHS = 12        # stage 2 maximum; patience 2 keeps the best epoch


In [ ]:
from pathlib import Path
from hashlib import sha256
import json, os, re, torch
assert torch.cuda.is_available(), "Enable a Kaggle GPU before running."
assert re.fullmatch(r"[A-Za-z0-9_-]+", RUN_NAME), "Use a simple run name."
matches = [p.parent for p in Path("/kaggle/input").rglob("package_receipt.json")
           if (p.parent / "train_dailydialog_qwen.py").is_file()]
assert len(matches) == 1, "Attach exactly one supplied private two-stage input dataset."
bundle = matches[0]
receipt = json.loads((bundle / "package_receipt.json").read_text())
assert receipt["experiment"] == "dailydialog_then_reviewed_qwen"
assert receipt.get("gradient_policy") == "finite_only_amp_retry_v1", "Update the attached dataset to its latest version for the FP16 overflow fix."
for name, expected in receipt["files_sha256"].items():
    assert Path(name).name == name, "Invalid package filename"
    assert sha256((bundle / name).read_bytes()).hexdigest() == expected, f"Changed input: {name}"
output = Path("/kaggle/working") / RUN_NAME
assert not output.exists(), "Choose a new RUN_NAME to preserve this run's results."
manifest = json.loads((bundle / "manifest.json").read_text())
print({"gpu": torch.cuda.get_device_name(0), "training_conversations": len(manifest["train_ids"]),
       "validation_conversations": len(manifest["validation_ids"]),
       "has_reviewed_alternatives": (bundle / "reviewed_pairs.json").is_file(), "output": str(output)})


In [ ]:
env = dict(os.environ, OMP_NUM_THREADS="2", MKL_NUM_THREADS="2", PYTHONUNBUFFERED="1")
subprocess.check_call([
    sys.executable, str(bundle / "train_dailydialog_qwen.py"),
    "--bundle", str(bundle), "--output", str(output),
    "--max-train-queries", str(MAX_TRAIN_QUERIES),
    "--max-validation-queries", str(MAX_VALIDATION_QUERIES),
    "--adaptation-epochs", str(ADAPTATION_EPOCHS),
    "--refinement-epochs", str(REFINEMENT_EPOCHS),
], env=env)


In [ ]:
result = json.loads((output / "experiment.json").read_text())
assert result["completed"] and result["production_checkpoint"]
first, final = result["adaptation"], result["refinement"]
assert first["base_unchanged"] and final["base_unchanged"]
assert final["initialization"]["adapter_sha256"] == first["adapter_sha256"]
assert final["initial_adapter_parameters_sha256"] == first["best_adapter_parameters_sha256"]
for name, stage in [("DailyDialog adaptation", first), ("Reviewed refinement", final)]:
    print(name, {k: stage[k] for k in ["epochs_completed", "best_epoch", "best_validation_loss", "optimizer_steps"]})
print("Final adapter:", output / result["final_adapter"])
print("Pair losses from different stages are not directly comparable. Reply quality still needs matched evaluation.")


In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, title, stage in zip(axes, ["DailyDialog adaptation", "Reviewed refinement"], [first, final]):
    history = stage["history"]
    ax.plot([h["epoch"] for h in history], [h["train_loss"] for h in history], label="Train")
    ax.plot([h["epoch"] for h in history], [h["validation_loss"] for h in history], label="Validation")
    ax.axvline(stage["best_epoch"], linestyle=":", color="gray", label="Selected epoch")
    ax.set(title=title, xlabel="Epoch", ylabel="Binary suitability loss")
    ax.legend()
fig.tight_layout()
fig.savefig(output / "training_curves.png", dpi=160)
plt.show()


In [ ]:
import shutil
from IPython.display import FileLink, display
# Includes both selected adapters, histories and provenance; base model weights are not copied.
archive = shutil.make_archive(str(output) + "_results", "zip", root_dir=output)
display(FileLink(Path(archive).name))


## After training
Download the results ZIP from the link or Kaggle Output. Keep `experiment.json`, both stage directories, `adaptation_pairs/manifest.json` and `training_curves.png` together. The final adapter is `stage2_reviewed/best/`; it needs the same pinned Qwen base model for inference.

Do not replace the current demo checkpoint yet. Compare the final adapter against raw Qwen, reviewed-only Qwen and V5 on the **same saved validation candidate pool**. Break results down by Standard/Local and repeated-speaker histories. Lower training loss is not evidence of better reply selection. The 60-item project test set stays untouched.

Source: [Li et al., 2017](https://aclanthology.org/I17-1099/) and [DailyDialog dataset licence](https://huggingface.co/datasets/li2017dailydialog/daily_dialog). Data mirror and archive hashes are recorded in the generated manifest.
